# PART 7-1. 리뷰 감성 분석 ① 머신러닝 (TF-IDF + 로지스틱 회귀)

> **핵심 질문: 별점 없이 글만 보고 불만을 알아낼 수 있을까?**

## 📋 한눈에 보기

**작업 흐름**
```
1. 통합 데이터에서 리뷰 중복 제거
2. 글 없는 리뷰 · 3점 제외 → 긍정/부정 라벨
3. train / test 분리
4. TF-IDF (train으로만 학습)
5. 로지스틱 회귀 (class_weight)
6. 평가 · 단어 해석 · 오분류 확인
7. bigram 개선 실험
8. 모델 저장
```

| 항목 | 내용 |
|---|---|
| 사용 데이터 | `olist_master_data.csv`의 리뷰 → 37,139건 (긍정 26,380 / 부정 10,759) |
| 결과물 | `part7_1_sentiment_ml.pkl` (약 170KB, **배포용 감성 모델**) |

**🎯 핵심 (꼭 가져갈 것)**
- 단어집(TF-IDF)도 **train으로만** 만들고, test는 변환만 합니다.
- 정확도만 보지 말고 **부정 리뷰의 재현율·F1**을 봅니다.

**💡 인사이트 (이 파트에서 확인한 사실)**
- 정확도 0.917 (전부 긍정이라고 찍으면 0.710), 부정 F1 0.869
- 부정 단어에서 **배송 지연 · 오배송 · 품질 · 파손**이라는 불만 유형이 드러납니다.
- 입력만 다듬어도(bigram) 정확도 0.926으로 오릅니다.

---

## 수업 목표
- 리뷰 문장을 **단어 빈도(TF-IDF)**로 숫자로 바꾸고, **로지스틱 회귀**로 긍정/부정을 분류합니다.
- 분류 평가 지표(Accuracy, Precision, Recall, F1, Confusion Matrix)를 이해합니다.
- 모델이 어떤 단어를 긍정·부정의 근거로 삼았는지 **직접 해석**해 봅니다.
- 같은 문제를 PART 7-2(딥러닝), PART 7-3(BERT)로 풀어 비교하는 출발점을 만듭니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 리뷰 1개 = 1행. 4~5점은 긍정, 1~2점은 부정, 3점은 제외합니다. |
| 코드 관점 | TfidfVectorizer, LogisticRegression, classification_report, confusion_matrix 사용법을 익힙니다. |
| AI 관점 | 머신러닝 분류의 기본 흐름입니다. 단순하지만 강력해서 실무에서 **첫 번째 기준 모델**로 많이 씁니다. |

## 왜 리뷰 감성을 분류할까?
| 쓰임 | 설명 |
|---|---|
| 부정 리뷰 자동 수집 | 리뷰가 수만 개라 사람이 다 읽을 수 없습니다. 부정 리뷰만 골라 CS팀에 전달할 수 있습니다. |
| 별점과 내용이 다른 리뷰 찾기 | 별점은 높은데 내용은 불만인 리뷰 등을 찾을 수 있습니다. |
| 별점이 없는 글에 적용 | 별점 달린 리뷰로 학습한 뒤, 별점이 없는 고객 문의·SNS 글의 불만을 찾는 데 씁니다. |
| PART 5와 연결 | 인사이트 9(지연 → 나쁜 리뷰)처럼, 부정 리뷰에 어떤 단어가 많은지 확인할 수 있습니다. |

---

## 📌 이 파트에서 사용하는 API 흐름 (scikit-learn)

| 단계 | 함수 | 역할 |
|---|---|---|
| ① | TfidfVectorizer.fit_transform() | 문장 → 단어 빈도 숫자 벡터 (train으로만 학습) |
| ② | LogisticRegression.fit() | 모델 학습 |
| ③ | predict() / predict_proba() | 예측 (긍정/부정, 긍정 확률) |
| ④ | classification_report / confusion_matrix | 평가 |

## 핵심 개념

| 개념 | 설명 |
|---|---|
| 분류(Classification) | 정해진 범주(긍정/부정) 중 하나를 예측하는 모델 |
| 토큰(Token) | 문장을 나눈 작은 단위. 여기서는 단어 |
| TF (Term Frequency) | 한 리뷰 안에서 어떤 단어가 얼마나 자주 나왔는가 |
| IDF (Inverse Document Frequency) | 많은 리뷰에 흔하게 나오는 단어(o, de 등)는 덜 중요하게, 드문 단어는 더 중요하게 보는 가중치 |
| TF-IDF | TF × IDF. 그 리뷰를 **잘 나타내는 단어**일수록 큰 값 |
| 로지스틱 회귀 | 단어마다 가중치를 곱해 더한 값을 0~1 확률로 바꿔 긍정/부정을 판단하는 분류 모델 (이름은 '회귀'지만 분류 모델) |
| class_weight | 수가 적은 클래스(부정)를 틀리면 더 큰 벌점을 줘서 불균형을 보정 |

In [1]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 셀 2. 라이브러리 불러오기

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

base_path  = "/content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist"
model_path = f"{base_path}/model"
os.makedirs(model_path, exist_ok=True)

In [3]:
# 셀 3. 데이터 불러오기

df = pd.read_csv(f"{base_path}/data/olist_master_data.csv")

# 통합 데이터는 주문 1건이 여러 행이라 같은 리뷰가 여러 번 들어 있습니다.
# review_id 기준으로 중복을 지워 리뷰 1개 = 1행으로 만듭니다.
review_df = df.drop_duplicates("review_id")[["review_comment_message", "review_score"]].copy()
print(f"전체 리뷰 수: {len(review_df):,}")

전체 리뷰 수: 98,411


In [4]:
# 셀 4. 데이터 전처리 + 라벨 만들기

review_df = review_df.dropna()                               # 리뷰 글이 없는 행 제거
review_df = review_df[review_df["review_score"] != 3]       # 중립(3점) 제거

# 4~5점 → 긍정(1),  1~2점 → 부정(0)
review_df["label"] = (review_df["review_score"] >= 4).astype(int)

print(f"분석할 리뷰 수: {len(review_df):,}")
print(review_df["label"].value_counts())
# ▶ 긍정이 부정보다 약 2.5배 많습니다. → 불균형 데이터

분석할 리뷰 수: 37,139
label
1    26380
0    10759
Name: count, dtype: int64


In [5]:
# 셀 5. train / test 분리
# PART 7-2·7-3과 똑같이 나눕니다. (random_state=42) → 세 모델을 같은 시험지로 비교할 수 있습니다.

X_train, X_test, y_train, y_test = train_test_split(
    review_df["review_comment_message"], review_df["label"],
    test_size=0.2, random_state=42,
    stratify=review_df["label"]    # 긍정/부정 비율을 train/test에 똑같이 유지
)
print(f"train: {len(X_train):,}  /  test: {len(X_test):,}")

train: 29,711  /  test: 7,428


---
## 문장을 숫자로: TF-IDF

```
리뷰 1: "produto muito bom"            리뷰 2: "não recebi o produto"
                ↓ TfidfVectorizer
            produto   muito   bom   não   recebi   o
리뷰 1        0.41    0.58   0.70   0      0      0
리뷰 2        0.33     0      0    0.56   0.62   0.44
```

> 각 리뷰가 **단어 개수만큼의 칸**을 가진 숫자 벡터가 됩니다. 단어 순서는 보지 않습니다. (Bag of Words)

In [6]:
# 셀 6. TF-IDF 벡터 만들기
# max_features=10000 : 자주 나오는 단어 10,000개만 사용 (PART 7-2 딥러닝과 같은 크기)
# ⚠️ fit은 train으로만! test는 transform만 합니다. (test의 단어 정보를 미리 보면 누수)

tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("train 벡터 크기:", X_train_tfidf.shape, "→ (리뷰 수, 단어 수)")
print("한 리뷰에 실제로 들어 있는 단어 수(평균):", round(X_train_tfidf.getnnz(axis=1).mean(), 1))
# ▶ 칸은 10,000개지만 한 리뷰에는 십여 개 단어만 있어서 대부분 0입니다. (희소 행렬)

train 벡터 크기: (29711, 10000) → (리뷰 수, 단어 수)
한 리뷰에 실제로 들어 있는 단어 수(평균): 9.4


### 🔵 로지스틱 회귀 학습

In [7]:
# 셀 7. 로지스틱 회귀 학습
# class_weight="balanced" : 적은 클래스(부정)를 틀리면 더 큰 벌점 → 불균형 보정

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_tfidf, y_train)
print("학습 완료!")

학습 완료!


In [8]:
# 셀 8. 평가

y_pred = model.predict(X_test_tfidf)

majority_acc = (y_test == 1).mean()     # 기준선 : 전부 '긍정'이라고 찍기
print(f"기준선 (전부 긍정) 정확도 : {majority_acc:.3f}")
print(f"로지스틱 회귀 정확도     : {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["부정(0)", "긍정(1)"], digits=3))
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))
print("  행 = 실제 [부정, 긍정] / 열 = 예측 [부정, 긍정]")

기준선 (전부 긍정) 정확도 : 0.710
로지스틱 회귀 정확도     : 0.917

              precision    recall  f1-score   support

       부정(0)      0.804     0.947     0.869      2152
       긍정(1)      0.976     0.906     0.940      5276

    accuracy                          0.917      7428
   macro avg      0.890     0.926     0.904      7428
weighted avg      0.926     0.917     0.919      7428

=== Confusion Matrix ===
[[2037  115]
 [ 498 4778]]
  행 = 실제 [부정, 긍정] / 열 = 예측 [부정, 긍정]


## 평가 지표 읽는 법

|  | 예측: 부정 | 예측: 긍정 |
|---|---|---|
| **실제: 부정** | 맞힘 (TN) | 놓침 (FN) |
| **실제: 긍정** | 잘못 경보 (FP) | 맞힘 (TP) |

| 지표 | 부정 리뷰 기준 의미 |
|---|---|
| Precision | "부정"이라고 고른 리뷰 중 실제 부정 비율 → 낮으면 CS팀이 쓸데없는 리뷰를 읽게 됨 |
| Recall | 실제 부정 리뷰 중 찾아낸 비율 → 낮으면 불만 고객을 놓침 |
| F1 | Precision과 Recall의 균형 점수 |

> 부정 리뷰를 놓치지 않는 것이 중요하다면 **부정 Recall**을 특히 봅니다.

## 🔍 모델은 어떤 단어로 판단했을까?

로지스틱 회귀는 **단어마다 가중치**를 배웁니다. 가중치가 큰 단어는 긍정 쪽으로, 작은(음수) 단어는 부정 쪽으로 판단을 끌어당깁니다.
딥러닝과 달리 **판단 근거를 직접 들여다볼 수 있다**는 것이 큰 장점입니다.

In [9]:
# 셀 9. 긍정·부정을 가장 강하게 나타내는 단어 TOP 15

words   = np.array(tfidf.get_feature_names_out())   # 단어 목록
weights = model.coef_[0]                             # 단어별 가중치

top_pos = np.argsort(weights)[::-1][:15]             # 가중치가 가장 큰 15개
top_neg = np.argsort(weights)[:15]                   # 가중치가 가장 작은 15개

word_table = pd.DataFrame({
    "긍정 단어": words[top_pos], "긍정 가중치": weights[top_pos].round(2),
    "부정 단어": words[top_neg], "부정 가중치": weights[top_neg].round(2),
})
word_table

,긍정 단어,긍정 가중치,부정 단어,부정 가중치
0,antes,6.54,não,-10.00
1,parabéns,4.94,nao,-5.41
2,ótimo,4.86,péssima,-4.67
3,rápida,4.78,ruim,-3.93
4,rápido,4.70,péssimo,-3.91
5,excelente,4.70,diferente,-3.85
6,testei,4.56,nem,-3.75
7,lindo,4.54,comprei,-3.74
8,perfeito,4.50,baixa,-3.68
9,amei,4.37,aguardando,-3.48


**해석 (출력된 단어의 뜻)**

| 구분 | 단어 | 뜻 | 묶어 보면 |
|---|---|---|---|
| 긍정 | antes · rápido · rápida | (기한) 전에 · 빠른 | **빠른 배송** |
| 긍정 | ótimo · ótima · excelente · perfeito · lindo | 훌륭한 · 탁월한 · 완벽한 · 예쁜 | 상품 칭찬 |
| 긍정 | amei · adorei · recomendo · satisfeito · parabéns | 정말 좋았다 · 추천한다 · 만족한 · 축하/칭찬 | 만족 표현 |
| 부정 | não · nao · nem | 아니다 (nao는 악센트 없이 쓴 같은 단어) · ~도 아니다 | 부정어 |
| 부정 | péssimo · péssima · ruim · baixa · inferior | 최악의 · 나쁜 · 낮은(품질) · 기대 이하 | 품질 불만 |
| 부정 | aguardando · demora · passou | 기다리는 중 · 오래 걸림 · (기한이) 지났다 | **배송 지연** |
| 부정 | diferente · comprei · errado | 다른 · 샀는데 · 잘못된 | **주문과 다른 상품 (오배송)** |
| 부정 | quebrado | 깨진 | 파손 |

> 💡 모델은 정답(별점)만 보고 학습했는데도, 부정 리뷰의 단어에서 **배송 지연 · 오배송 · 품질 · 파손**이라는 불만 유형이 드러납니다.
> 특히 배송 관련 단어가 긍정(빠른 배송)과 부정(지연) 양쪽에 모두 나타납니다. → PART 5 인사이트 9(배송이 늦으면 리뷰가 무너진다)와 같은 이야기입니다.

In [10]:
# 셀 10. 새 문장 예측

def predict_sentiment(text):
    """문장 → 긍정/부정과 긍정 확률"""
    prob = model.predict_proba(tfidf.transform([text]))[0, 1]
    return ("긍정" if prob >= 0.5 else "부정"), round(float(prob), 3)

examples = [
    "produto muito bom, chegou antes do prazo",     # 상품 아주 좋음, 기한보다 일찍 도착
    "ainda não recebi o produto",                   # 아직 상품을 받지 못했다
    "produto veio quebrado, péssimo",               # 상품이 깨져서 왔다, 최악
]
for text in examples:
    print(f"{text:45s} → {predict_sentiment(text)}")

produto muito bom, chegou antes do prazo      → ('긍정', 0.999)
ainda não recebi o produto                    → ('부정', 0.008)
produto veio quebrado, péssimo                → ('부정', 0.006)


In [11]:
# 셀 11. 틀린 예측 살펴보기 — 모델은 어디서 헷갈릴까?

result = pd.DataFrame({"text": X_test.values, "label": y_test.values, "pred": y_pred})
wrong = result[result["label"] != result["pred"]]
print(f"틀린 리뷰: {len(wrong):,}개 / {len(result):,}개")
print()
for _, row in wrong.sample(5, random_state=1).iterrows():
    print(f"[실제 {'긍정' if row.label == 1 else '부정'} → 예측 {'긍정' if row.pred == 1 else '부정'}] {row.text[:90]}")
# ▶ 별점과 내용이 다른 리뷰, 부정어가 섞인 긍정 리뷰("não tive problema" = 문제없었다) 등에서 자주 틀립니다.
#   단어 순서를 보지 않는 TF-IDF의 한계입니다. → PART 7-2·7-3에서 다른 방법으로 비교합니다.

틀린 리뷰: 613개 / 7,428개

[실제 긍정 → 예측 부정] Recebi o sabor errado, comprei morango e veio chocolate
[실제 부정 → 예측 긍정] Comprei Cinta Térmica Ajustável Modeladora Abdominal Shape Now HB,porém chegou Cinta Model
[실제 긍정 → 예측 부정] Entrega rápida e relogio muito bonito. Porém eu comprei com os números na cor azul e veio 
[실제 부정 → 예측 긍정] sem opinião
[실제 긍정 → 예측 부정] só não gostei por q o produto não tem o selo do imetro. mais o resto estava tudo serto.


---
## 🔧 개선 실험: 2단어 묶음(bigram) + 악센트 통일

기본 모델은 **단어 하나하나**만 봅니다. 그래서 "não gostei(좋지 않았다)"와 "gostei(좋았다)"를 구분하기 어렵습니다.
TF-IDF 설정만 바꿔서 이 한계를 줄여 봅니다.

| 설정 | 의미 |
|---|---|
| `ngram_range=(1, 2)` | 1단어 + **붙어 있는 2단어 묶음**도 특징으로 사용 (예: "não gostei") |
| `strip_accents="unicode"` | "não"와 "nao"처럼 악센트만 다른 단어를 같은 단어로 통일 |
| `sublinear_tf=True` | 같은 단어를 여러 번 반복해도 영향이 너무 커지지 않게 조절 |
| `max_features=30000` | 2단어 묶음이 추가되어 단어집 크기를 늘림 |

> 📌 저장하는 모델(셀 12)은 **기본 모델 그대로** 둡니다. PART 7-2·7-3에서 같은 기준으로 비교하기 위해서입니다.

In [12]:
# 셀 11-1. 개선 모델 : bigram + 악센트 통일
# ⚠️ 이번에도 fit은 train으로만, test는 transform만 합니다.

tfidf2 = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),        # 1단어 + 2단어 묶음
    sublinear_tf=True,         # 반복 단어의 영향 줄이기
    strip_accents="unicode",   # não ↔ nao 통일
)
X_train_tfidf2 = tfidf2.fit_transform(X_train)
X_test_tfidf2  = tfidf2.transform(X_test)

model2 = LogisticRegression(max_iter=2000, class_weight="balanced", C=2)
# C : 값이 클수록 train 데이터에 더 맞추려고 합니다. (기본값 1)
model2.fit(X_train_tfidf2, y_train)
y_pred2 = model2.predict(X_test_tfidf2)

print(f"기본 모델   정확도 : {accuracy_score(y_test, y_pred):.3f}")
print(f"bigram 모델 정확도 : {accuracy_score(y_test, y_pred2):.3f}")
print()
print(classification_report(y_test, y_pred2, target_names=["부정(0)", "긍정(1)"], digits=3))

# 부정어가 들어간 문장으로 두 모델 비교
for text in ["não gostei", "não tive problema nenhum, chegou rápido"]:
    p1 = model.predict_proba(tfidf.transform([text]))[0, 1]
    p2 = model2.predict_proba(tfidf2.transform([text]))[0, 1]
    print(f"{text:42s} 긍정 확률  기본 {p1:.3f}  /  bigram {p2:.3f}")
# ▶ 설정 몇 개만 바꿔도 성능이 오릅니다. 모델을 바꾸기 전에 '입력(특징)'부터 다듬는 것이 실무의 기본입니다.

기본 모델   정확도 : 0.917
bigram 모델 정확도 : 0.926

              precision    recall  f1-score   support

       부정(0)      0.826     0.943     0.880      2152
       긍정(1)      0.975     0.919     0.946      5276

    accuracy                          0.926      7428
   macro avg      0.901     0.931     0.913      7428
weighted avg      0.932     0.926     0.927      7428

não gostei                                 긍정 확률  기본 0.060  /  bigram 0.004
não tive problema nenhum, chegou rápido    긍정 확률  기본 0.774  /  bigram 0.821


---
## 💾 모델 저장: TF-IDF와 모델을 함께 저장하는 이유

모델은 **숫자**만 이해합니다. 새 문장을 예측하려면 **학습 때와 똑같은 단어집(TF-IDF)**으로 숫자로 바꿔야 합니다. (셀 10 `predict_sentiment` 함수가 바로 이 순서입니다)

```
새 문장 → tfidf.transform()  (학습 때 만든 단어집으로 변환) → model.predict_proba() → 긍정 확률
```

| 실수 | 결과 |
|---|---|
| 모델만 저장하고 TF-IDF는 저장 안 함 | 새 문장을 숫자로 바꿀 방법이 없음 |
| 새 문장으로 TF-IDF를 다시 `fit` | 단어 번호가 학습 때와 달라져 **엉뚱한 예측** |

> 📌 그래서 `{"tfidf": ..., "model": ...}`처럼 **한 파일에 함께** 저장합니다. STEP2(서빙)에서 이 파일 하나만 불러오면 됩니다.

In [13]:
# 셀 12. 모델 저장 (STEP2에서 서빙에 사용)
# 문장을 숫자로 바꾸는 tfidf와 모델을 함께 저장해야 새 문장을 예측할 수 있습니다.

# compress=3 : 압축해서 저장 → 파일 크기가 약 1/3로 줄어듭니다. (불러올 때는 joblib.load 그대로 사용)
joblib.dump({"tfidf": tfidf, "model": model}, f"{model_path}/part7_1_sentiment_ml.pkl", compress=3)
print("감성 분석(머신러닝) 모델 저장 완료!")
print("저장 위치:", f"{model_path}/part7_1_sentiment_ml.pkl")
print(f"파일 크기: {os.path.getsize(f'{model_path}/part7_1_sentiment_ml.pkl') / 1024:.0f} KB")


감성 분석(머신러닝) 모델 저장 완료!
저장 위치: /content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist/model/part7_1_sentiment_ml.pkl
파일 크기: 176 KB


---

## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | TF-IDF로 문장을 숫자로 바꾸고, 로지스틱 회귀로 긍정/부정을 분류했습니다. |
| 누수 방지 | 단어집(TF-IDF)은 train으로만 만들고, test는 변환만 합니다. |
| 평가 | 정확도만 보지 않고 부정 리뷰의 Precision·Recall과 Confusion Matrix를 함께 봅니다. |
| 해석 | 로지스틱 회귀는 단어별 가중치로 판단 근거를 직접 확인할 수 있습니다. |
| 한계 | 단어 순서를 보지 않아 부정어·문맥을 놓칠 수 있습니다. |
| 개선 | 2단어 묶음(bigram)·악센트 통일처럼 입력을 다듬으면 같은 모델로도 성능이 오릅니다. |
| 모델 저장 | 단어집(TF-IDF)과 모델을 한 파일에 함께, 압축(compress=3)해서 저장합니다. |
| 다음 단계 | PART 7-2에서 같은 리뷰를 딥러닝(Embedding 신경망)으로 분류해 비교합니다. |